# Predictive Analytics: Trivial Demand Baselines

This notebook fits three leakage-free baselines on each training split and applies them separately to the corresponding validation and test splits:

1. always predict zero;
2. predict the global median demand observed in the training split;
3. predict the training mean for the same spatial unit, weekday, and time bucket.

Validation and test targets are never used to estimate a baseline. If a spatial/weekday/time combination was not observed during training, the third baseline predicts zero. Predictions are persisted with their timestamp and spatial identifier so model notebooks can compare results on exactly the same observations.

In [1]:
from dataclasses import dataclass
from pathlib import Path

import polars as pl

from helper_functions import compute_regression_metrics
from run_config import PATHS, demand_split_paths

## Dataset configuration

The comparison includes H3 resolution 7, census tracts, filtered community areas, and the separate unfiltered community-area datasets for all three time units.

In [2]:
# H3 resolution 7 is explicit so paths, tags and output files cannot silently
# change when the default resolution in run_config changes. The unfiltered
# Community Area datasets are separate experiments with their own model tags.
DATASET_CONFIGS = (
    {"spatial_unit": "hexagon", "spatial_col": "h3_cell", "time_unit": "1h", "h3_resolution": 7},
    {"spatial_unit": "hexagon", "spatial_col": "h3_cell", "time_unit": "1h", "h3_resolution": 8},
    {"spatial_unit": "census_tracts", "spatial_col": "census_tract", "time_unit": "1h"},
    {"spatial_unit": "community_areas", "spatial_col": "community_area", "time_unit": "1h"},
    {"spatial_unit": "community_areas_unfiltered", "spatial_col": "community_area", "time_unit": "1h"},
    {"spatial_unit": "hexagon", "spatial_col": "h3_cell", "time_unit": "24h", "h3_resolution": 7},
    {"spatial_unit": "hexagon", "spatial_col": "h3_cell", "time_unit": "24h", "h3_resolution": 8},
    {"spatial_unit": "census_tracts", "spatial_col": "census_tract", "time_unit": "24h"},
    {"spatial_unit": "community_areas", "spatial_col": "community_area", "time_unit": "24h"},
    {"spatial_unit": "community_areas_unfiltered", "spatial_col": "community_area", "time_unit": "24h"},
    {"spatial_unit": "hexagon", "spatial_col": "h3_cell", "time_unit": "4h", "h3_resolution": 7},
    {"spatial_unit": "hexagon", "spatial_col": "h3_cell", "time_unit": "4h", "h3_resolution": 8},
    {"spatial_unit": "census_tracts", "spatial_col": "census_tract", "time_unit": "4h"},
    {"spatial_unit": "community_areas", "spatial_col": "community_area", "time_unit": "4h"},
    {"spatial_unit": "community_areas_unfiltered", "spatial_col": "community_area", "time_unit": "4h"},
)

TARGET_COL = "trip_count"
TIME_COL = "datetime_hour"
PREDICTION_DIR = PATHS.train_test_dir / "baseline_predictions"
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)


def split_paths_for(dataset_config):
    spatial_unit = dataset_config["spatial_unit"]
    time_unit = dataset_config["time_unit"]
    if spatial_unit == "hexagon":
        return demand_split_paths(
            spatial_unit, time_unit,
            h3_resolution=dataset_config["h3_resolution"],
        )
    if spatial_unit == "community_areas_unfiltered":
        # demand_split_paths currently exposes only the filtered Community
        # Area variant, so derive these split paths from its configured Gold path.
        gold_path = getattr(
            PATHS, f"gold_{time_unit}_demand_community_area_unfiltered"
        )
        return {
            split: PATHS.train_test_dir / f"{gold_path.stem}_{split.upper()}.parquet"
            for split in ("train", "val", "test")
        }
    return demand_split_paths(spatial_unit, time_unit)


def model_tag_for(dataset_config):
    if dataset_config["spatial_unit"] == "hexagon":
        return (
            f"hexagon_h3r{dataset_config['h3_resolution']}_"
            f"{dataset_config['time_unit']}"
        )
    return f"{dataset_config['spatial_unit']}_{dataset_config['time_unit']}"

## Fit and prediction functions

`fit_baselines` derives all parameters from a training split. `predict_validation_baselines` and `predict_test_baselines` create the three predictions for the matching evaluation split without refitting.

In [3]:
@dataclass(frozen=True)
class FittedBaselines:
    model_tag: str
    spatial_col: str
    global_median: float
    spatial_time_weekday_means: pl.DataFrame


def validate_baseline_frame(frame: pl.DataFrame, spatial_col: str, split: str) -> None:
    required = {TIME_COL, TARGET_COL, spatial_col, "weekday", "hour"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{split} split is missing columns: {sorted(missing)}")
    if frame.is_empty():
        raise ValueError(f"{split} split is empty")
    if frame[TARGET_COL].null_count():
        raise ValueError(f"{split} split contains null targets")


def fit_baselines(dataset_config: dict[str, object]) -> FittedBaselines:
    spatial_unit = dataset_config["spatial_unit"]
    spatial_col = dataset_config["spatial_col"]
    time_unit = dataset_config["time_unit"]
    split_paths = split_paths_for(dataset_config)
    train_df = pl.read_parquet(split_paths["train"])
    validate_baseline_frame(train_df, spatial_col, "train")

    global_median = float(train_df[TARGET_COL].median())
    group_cols = [spatial_col, "weekday", "hour"]
    historical_means = (
        train_df
        .group_by(group_cols)
        .agg(
            pl.col(TARGET_COL)
            .mean()
            .alias("prediction_spatial_time_weekday_mean")
        )
    )
    return FittedBaselines(
        model_tag=model_tag_for(dataset_config),
        spatial_col=spatial_col,
        global_median=global_median,
        spatial_time_weekday_means=historical_means,
    )


def predict_baselines(frame: pl.DataFrame, fitted: FittedBaselines) -> pl.DataFrame:
    """Apply fitted baselines to a frame without using its targets for fitting."""
    validate_baseline_frame(frame, fitted.spatial_col, "prediction")
    group_cols = [fitted.spatial_col, "weekday", "hour"]
    return (
        frame
        .join(fitted.spatial_time_weekday_means, on=group_cols, how="left")
        .with_columns(
            pl.lit(0.0).alias("prediction_zero"),
            pl.lit(fitted.global_median).alias("prediction_global_median"),
            pl.col("prediction_spatial_time_weekday_mean")
            .fill_null(0.0),
        )
        .select(
            TIME_COL,
            fitted.spatial_col,
            pl.col(TARGET_COL).alias("y_true"),
            "prediction_zero",
            "prediction_global_median",
            "prediction_spatial_time_weekday_mean",
        )
        .sort([TIME_COL, fitted.spatial_col])
    )


def predict_split_baselines(
    dataset_config: dict[str, object], split: str, *, save: bool = True
) -> pl.DataFrame:
    """Fit on TRAIN and predict either VAL or TEST without fitting on targets."""
    if split not in {"val", "test"}:
        raise ValueError("split must be either 'val' or 'test'")
    fitted = fit_baselines(dataset_config)
    split_paths = split_paths_for(dataset_config)
    prediction_df = pl.read_parquet(split_paths[split])
    validate_baseline_frame(prediction_df, fitted.spatial_col, split)
    predictions = predict_baselines(prediction_df, fitted)
    if predictions.height != prediction_df.height:
        raise ValueError(
            f"Prediction join changed row count: {prediction_df.height} -> {predictions.height}"
        )

    if save:
        output_path = PREDICTION_DIR / f"{fitted.model_tag}_{split}_baselines.parquet"
        predictions.write_parquet(output_path)
        print(f"Saved {predictions.height:,} {split} predictions to {output_path}")
    return predictions


def predict_validation_baselines(
    dataset_config: dict[str, object], *, save: bool = True
) -> pl.DataFrame:
    return predict_split_baselines(dataset_config, "val", save=save)


def predict_test_baselines(
    dataset_config: dict[str, object], *, save: bool = True
) -> pl.DataFrame:
    return predict_split_baselines(dataset_config, "test", save=save)

## Validation/test predictions and baseline metrics

The validation and test prediction files are reusable comparison artifacts. A separate metric table is saved for each split with the same regression helper used by the neural-network notebooks.

In [4]:
PREDICTION_COLUMNS = {
    "zero": "prediction_zero",
    "global_median": "prediction_global_median",
    "spatial_time_weekday_mean": "prediction_spatial_time_weekday_mean",
}

baseline_predictions = {"val": {}, "test": {}}
metric_tables = {"val": [], "test": []}

for dataset_config in DATASET_CONFIGS:
    model_tag = model_tag_for(dataset_config)
    for split in ("val", "test"):
        # Both artifacts use parameters fitted on TRAIN. VAL can therefore be
        # consumed during model development while TEST remains for final evaluation.
        predictions = predict_split_baselines(dataset_config, split)
        baseline_predictions[split][model_tag] = predictions

        for baseline_name, prediction_col in PREDICTION_COLUMNS.items():
            evaluation = predictions.select(
                "y_true", pl.col(prediction_col).alias("y_pred")
            )
            metric_tables[split].append(
                compute_regression_metrics(evaluation)
                .with_columns(
                    pl.lit(model_tag).alias("dataset"),
                    pl.lit(baseline_name).alias("baseline"),
                )
                .select("dataset", "baseline", "metric", "value")
            )

baseline_metrics = {}
for split in ("val", "test"):
    baseline_metrics[split] = pl.concat(metric_tables[split])
    metrics_path = PREDICTION_DIR / f"baseline_{split}_metrics.parquet"
    baseline_metrics[split].write_parquet(metrics_path)
    print(f"Saved {split} baseline metrics to {metrics_path}")

baseline_metrics["val"].filter(
    pl.col("metric").is_in(["mae", "rmse", "r2", "nonzero_mae", "peak_mae"])
).sort(["dataset", "metric", "value"])

Saved 285,576 val predictions to C:\Users\angel\git\Group-3-AAA\data\full\train_test_data\baseline_predictions\hexagon_h3r7_1h_val_baselines.parquet
Saved 285,576 test predictions to C:\Users\angel\git\Group-3-AAA\data\full\train_test_data\baseline_predictions\hexagon_h3r7_1h_test_baselines.parquet
Saved 1,709,952 val predictions to C:\Users\angel\git\Group-3-AAA\data\full\train_test_data\baseline_predictions\hexagon_h3r8_1h_val_baselines.parquet
Saved 1,709,952 test predictions to C:\Users\angel\git\Group-3-AAA\data\full\train_test_data\baseline_predictions\hexagon_h3r8_1h_test_baselines.parquet
Saved 1,538,256 val predictions to C:\Users\angel\git\Group-3-AAA\data\full\train_test_data\baseline_predictions\census_tracts_1h_val_baselines.parquet
Saved 1,538,256 test predictions to C:\Users\angel\git\Group-3-AAA\data\full\train_test_data\baseline_predictions\census_tracts_1h_test_baselines.parquet
Saved 134,904 val predictions to C:\Users\angel\git\Group-3-AAA\data\full\train_test_data\

dataset,baseline,metric,value
str,str,str,f64
"""census_tracts_1h""","""spatial_time_weekday_mean""","""mae""",0.140169
"""census_tracts_1h""","""zero""","""mae""",0.366031
"""census_tracts_1h""","""global_median""","""mae""",0.366031
"""census_tracts_1h""","""spatial_time_weekday_mean""","""nonzero_mae""",5.644907
"""census_tracts_1h""","""global_median""","""nonzero_mae""",16.733008
…,…,…,…
"""hexagon_h3r8_4h""","""zero""","""r2""",-0.005444
"""hexagon_h3r8_4h""","""spatial_time_weekday_mean""","""r2""",0.908596
"""hexagon_h3r8_4h""","""spatial_time_weekday_mean""","""rmse""",5.396812


## Reuse in later notebooks

A later model notebook can either call `predict_test_baselines(dataset_config, save=False)` after importing/running these definitions, or read the persisted file from `PATHS.train_test_dir / "baseline_predictions"`. Join model predictions to baseline predictions by `datetime_hour` and the corresponding spatial identifier before comparing errors.